In [1]:
#5. RAG
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama

# 1) Load the same embedding model you used when indexing
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

# 2) Load the persisted vector store
vectorstore = Chroma(
    persist_directory="./chroma_langchain_db",
    collection_name="uia_courses",
    embedding_function=embeddings,
)

# 3) Turn it into a retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# 4) Load your Ollama LLM
llm = ChatOllama(
    model="qwen2.5:0.5b",   # or another model you actually pulled
    temperature=0,
    base_url="http://localhost:11434"
)

# 5) Ask a question
question = "What are the learning outcomes of the deep neural network course?"

# Retrieve relevant chunks
docs = retriever.invoke(question)

# Build context
context = "\n\n".join(doc.page_content for doc in docs)

# Prompt the LLM
prompt = f"""
Answer the question using only the context below.
If the answer is not in the context, say you do not know.

Context:
{context}

Question:
{question}
"""

response = llm.invoke(prompt)

print("QUESTION:")
print(question)
print("\nRETRIEVED DOCUMENTS:")
for i, doc in enumerate(docs, 1):
    print(f"\n--- Document {i} ---")
    print(doc.page_content)
    print("Metadata:", doc.metadata)

print("\nANSWER:")
print(response.content)

/opt/miniconda3/envs/ikt469project/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


QUESTION:
What are the learning outcomes of the deep neural network course?

RETRIEVED DOCUMENTS:

--- Document 1 ---
Course Title: IKT469 Deep Neural Networks (Spring 2026)
URL: https://www.uia.no/english/studies/courses/2026/spring/ikt469.html
Section: Learning Outcomes
Metadata: {'course_leader': 'Morten Goodwin', 'lecture_semester': 'Spring', 'url': 'https://www.uia.no/english/studies/courses/2026/spring/ikt469.html', 'teaching_language': 'English', 'course_leaders': '', 'row_id': 22, 'responsible_department': 'Faculty of Engineering and Science', 'section': 'learning_outcomes', 'duration': '½ year', 'ects_credits': '7.5', 'title': 'IKT469 Deep Neural Networks (Spring 2026)'}

--- Document 2 ---
Course Title: IKT469 Deep Neural Networks (Spring 2026)
URL: https://www.uia.no/english/studies/courses/2026/spring/ikt469.html
Section: Teaching And Learning Methods

Combination of lectures, assignments, paper studies, lab, and report writing. The tasks are done individually or in small g